# 🏛️ LegalSaathi — Multi-Model RAG Pipeline


---
## 🔧 Phase 0 — Setup & Configuration


In [ ]:
# Cell 0.1 — Install packages
# google-generativeai = Gemini SDK (replaces openai)
!pip install -q google-genai pinecone rank-bm25 networkx langdetect tqdm groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 14.2 MB/s eta 0:00:00


In [ ]:
# Cell 0.2 — Mount Drive (persists BM25 cache between sessions)
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/LegalSaathi', exist_ok=True)
print("✓ Drive mounted")


Mounted at /content/drive
✓ Drive mounted


In [ ]:
# DIAGNOSTIC — find the correct model name for your account
print("Available embedding models:")
for m in gemini_client.models.list():
    if 'embed' in m.name.lower():
        print(f"  {m.name}")

Available embedding models:
  models/gemini-embedding-001
  models/gemini-embedding-2-preview
  models/gemini-embedding-2


In [ ]:
# Diagnostic — find exact generation model names
print("Available generation models:")
for m in gemini_client.models.list():
    if 'generate' in str(m.supported_actions).lower():
        print(f"  {m.name}")


Available generation models:
  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-4-26b-a4b-it
  models/gemma-4-31b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3.1-flash-lite
  models/gemini-3-pro-image-preview
  models/gemini-3-pro-image
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/gemini-3.1-flash-image
  models/gemini-3.1-flash-lite-image
  models/gemini-3.5-flash
  models/gemini-omni-flash-preview
  models/lyria-3-clip-preview
  models/lyria-

In [ ]:
# Cell 0.3 — API Keys + Config
from google.colab import userdata
from google import genai
from google.genai import types

GEMINI_KEY   = userdata.get('GEMINI_API_KEY')
PINECONE_KEY = userdata.get('PINECONE_API_KEY')

gemini_client = genai.Client(api_key=GEMINI_KEY)
embed_client  = genai.Client(
    api_key      = GEMINI_KEY,
    http_options = types.HttpOptions(api_version='v1')
)

from groq import Groq
GROQ_KEY    = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=GROQ_KEY)

class Config:
    CHUNKS_PATHS = [
        '/content/constitution_rag_chunks.ndjson',
        '/content/crpc_rag_chunks.ndjson',
        '/content/ipc_rag_chunks.ndjson',
        '/content/pocso_rag_chunks.ndjson',
        '/content/dvact_rag_chunks.ndjson',          # was: pwdva_rag_chunks
        '/content/orissa_rent_rag_chunks.ndjson',    # was: orissa_house_rent_control_act_chunks
        '/content/mpl_rag_chunks.ndjson',            # was: muslim_personal_law_chunks
        '/content/ssc_rag_chunks.ndjson',            # was: social_security_code_chunks
        '/content/sma_rag_chunks.ndjson',            # was: special_marriage_act_chunks
        '/content/consumer_protection_act_chunks.ndjson',
    ]
    GRAPH_NODES_PATHS = [
        '/content/constitution_graph_nodes.ndjson',
        '/content/crpc_graph_nodes.ndjson',
        '/content/ipc_graph_nodes.ndjson',
        '/content/pocso_graph_nodes.ndjson',
        '/content/dvact_graph_nodes.ndjson',         # was: pwdva_graph_nodes
        '/content/orissa_rent_graph_nodes.ndjson',   # was: orissa_house_rent_control_act_graph_nodes
        '/content/mpl_graph_nodes.ndjson',           # was: muslim_personal_law_graph_nodes
        '/content/ssc_graph_nodes.ndjson',           # was: social_security_code_graph_nodes
        '/content/sma_graph_nodes.ndjson',           # was: special_marriage_act_graph_nodes
        '/content/consumer_protection_act_graph_nodes.ndjson',
    ]
    GRAPH_EDGES_PATHS = [
        '/content/constitution_graph_edges.ndjson',
        '/content/crpc_graph_edges.ndjson',
        '/content/ipc_graph_edges.ndjson',
        '/content/pocso_graph_edges.ndjson',
        '/content/dvact_graph_edges.ndjson',         # was: pwdva_graph_edges
        '/content/orissa_rent_graph_edges.ndjson',   # was: orissa_house_rent_control_act_graph_edges
        '/content/mpl_graph_edges.ndjson',           # was: muslim_personal_law_graph_edges
        '/content/ssc_graph_edges.ndjson',           # was: social_security_code_graph_edges
        '/content/sma_graph_edges.ndjson',           # was: special_marriage_act_graph_edges
        '/content/consumer_protection_act_graph_edges.ndjson',
    ]
    BM25_CACHE       = '/content/drive/MyDrive/LegalSaathi/bm25_index.pkl'

    EMBED_MODEL  = 'models/gemini-embedding-001'
    EMBED_DIM    = 3072
    LLM_MODEL    = 'llama-3.3-70b-versatile'

    PINECONE_API_KEY = PINECONE_KEY
    PINECONE_INDEX   = 'legal-saathi-gemini'
    PINECONE_NS      = 'all-acts'

    BM25_TOP_K        = 15
    DENSE_TOP_K       = 15
    FINAL_TOP_K       = 8
    GRAPH_HOP_DEPTH   = 2
    GRAPH_MAX_ADD     = 5
    MULTIHOP_ITERS    = 3
    MAX_CONTEXT_CHARS = 12_000

config = Config()
print(f"  Generation client : v1beta  ({config.LLM_MODEL})")
print(f"  Embedding client  : v1      ({config.EMBED_MODEL})")
print(f"  Acts configured   : {len(config.CHUNKS_PATHS)}")

  Generation client : v1beta  (llama-3.3-70b-versatile)
  Embedding client  : v1      (models/gemini-embedding-001)
  Acts configured   : 10


In [ ]:
# Cell 0.4 — Global imports
import json, pickle, os, re, time
from typing import List, Dict, Optional, Tuple, Set
from tqdm.notebook import tqdm
import networkx as nx
from rank_bm25 import BM25Okapi

# ── Universal entity-ID helper ─────────────────────────────────────────────
# Constitution chunks use 'article_number'; all other acts use 'section_number_raw'.
# Using this helper everywhere means we never hard-code 'article_number' again.
def get_entity_id(chunk: Dict) -> str:
    return (chunk.get('article_number') or
            chunk.get('section_number_raw') or
            chunk.get('section_number') or '')

print("✓ Imports ready")

✓ Imports ready


---
## 📂 Phase 1 — Data Loading & Exploration


In [ ]:
# Cell 1.1 — Upload NDJSON files
# Upload all 30 files (chunks + graph_nodes + graph_edges for each of the 10 acts)
from google.colab import files
print("Select all 30 NDJSON files (3 per act × 10 acts)...")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'/content/{name}', 'wb') as f: f.write(data)
    print(f"  Saved: /content/{name}  ({len(data):,} bytes)")

# Option B — from Drive (recommended after first upload):
# !cp /content/drive/MyDrive/LegalSaathi/*.ndjson /content/

Select all 30 NDJSON files (3 per act × 10 acts)...


Saving constitution_graph_edges.ndjson to constitution_graph_edges (1).ndjson
Saving constitution_graph_nodes.ndjson to constitution_graph_nodes (1).ndjson
Saving constitution_rag_chunks.ndjson to constitution_rag_chunks (1).ndjson
Saving crpc_graph_edges.ndjson to crpc_graph_edges (1).ndjson
Saving crpc_graph_nodes.ndjson to crpc_graph_nodes (1).ndjson
Saving crpc_rag_chunks.ndjson to crpc_rag_chunks (1).ndjson
Saving dvact_graph_edges.ndjson to dvact_graph_edges (1).ndjson
Saving dvact_graph_nodes.ndjson to dvact_graph_nodes (1).ndjson
Saving dvact_rag_chunks.ndjson to dvact_rag_chunks (1).ndjson
Saving hma_graph_edges.ndjson to hma_graph_edges (1).ndjson
Saving hma_graph_nodes.ndjson to hma_graph_nodes (1).ndjson
Saving hma_rag_chunks.ndjson to hma_rag_chunks (1).ndjson
Saving ipc_graph_edges.ndjson to ipc_graph_edges (1).ndjson
Saving ipc_graph_nodes.ndjson to ipc_graph_nodes (1).ndjson
Saving ipc_rag_chunks.ndjson to ipc_rag_chunks (1).ndjson
Saving mpl_graph_edges.ndjson to mpl_g

In [ ]:
# Cell 1.2 — Schema + statistics

# Load first act (Constitution) to display schema
with open(config.CHUNKS_PATHS[0], 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]

chunks_raw = [json.loads(l) for l in lines]
active     = [c for c in chunks_raw if not c.get('is_omitted', False)]
sample     = active[0]

print(f"Schema (from {config.CHUNKS_PATHS[0].split('/')[-1]}):")
for k, v in sample.items():
    print(f"  {k:<32} {type(v).__name__:<8} {str(v)[:55]}")

# Aggregate stats across all available act files
print("\n─── Chunk counts per act ───")
total = 0
for path in config.CHUNKS_PATHS:
    if not os.path.exists(path):
        print(f"  ⚠ NOT FOUND: {path.split('/')[-1]}"); continue
    with open(path, 'r', encoding='utf-8') as f:
        n = sum(1 for l in f if l.strip())
    total += n
    print(f"  {path.split('/')[-1]:<55} {n:>5} chunks")
print(f"  {'TOTAL':<55} {total:>5} chunks")

uc_counts = {}
for c in active:
    for uc in c.get('legalsaathi_use_cases', []):
        uc_counts[uc] = uc_counts.get(uc, 0) + 1
print("\nTop use_cases (Constitution sample):")
for uc, n in sorted(uc_counts.items(), key=lambda x:-x[1])[:10]:
    print(f"  {uc:<30} {n:>3}")

Schema (from constitution_rag_chunks.ndjson):
  chunk_id                         str      CONST_PREAMBLE
  document                         str      Constitution of India
  act_name                         str      The Constitution of India
  section_id                       str      Preamble
  section_number                   str      Preamble
  document_version                 str      104th Amendment (9 December 2020)
  chunk_type                       str      preamble
  part_id                          NoneType None
  part_title                       NoneType None
  article_number                   NoneType None
  title                            str      Preamble
  body_text                        str      P REAMB LE WE, THE PEOPLE OF INDIA, having solemnly res
  keywords                         list     ['SOVEREIGN', 'SOCIALIST', 'SECULAR', 'DEMOCRATIC', 'RE
  categories                       list     ['constitutional', 'preamble']
  rights_subtype                   NoneType Non

---
## 🔑 Phase 2 — BM25 Keyword Index
No API calls. Runs fully local. Cached to Drive after first build.


In [ ]:
# Cell 2.1 — LegalIndexer

class LegalIndexer:
    def __init__(self):
        self.chunks: List[Dict] = []
        self.bm25: Optional[BM25Okapi] = None
        self.id_to_idx: Dict[str, int] = {}
        self.art_to_ids: Dict[str, List[str]] = {}

    def load(self) -> None:
        self._load_chunks()
        self._build_art_map()
        if os.path.exists(config.BM25_CACHE):
            print("[BM25] Restoring from Drive cache...")
            with open(config.BM25_CACHE, 'rb') as f: self.bm25 = pickle.load(f)
        else:
            print("[BM25] Building index (first run)...")
            self._build()
            with open(config.BM25_CACHE, 'wb') as f: pickle.dump(self.bm25, f)
        print(f"[BM25] Ready — {len(self.chunks):,} chunks")

    def bm25_search(self, query: str, top_k: int = 15) -> List[Tuple[Dict, float]]:
        scores  = self.bm25.get_scores(query.lower().split())
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [(self.chunks[i], float(scores[i])) for i in top_idx if scores[i] > 0]

    def get_chunk(self, cid: str) -> Optional[Dict]:
        idx = self.id_to_idx.get(cid)
        return self.chunks[idx] if idx is not None else None

    def get_chunks_for_article(self, art: str) -> List[Dict]:
        return [self.get_chunk(c) for c in self.art_to_ids.get(art, []) if self.get_chunk(c)]

    def all_chunks(self) -> List[Dict]: return self.chunks

    def _load_chunks(self) -> None:
        # Load from ALL act chunk files
        for path in config.CHUNKS_PATHS:
            if not os.path.exists(path):
                print(f"  [Skip] {path.split('/')[-1]} not found")
                continue
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    c = json.loads(line)
                    if c.get('is_omitted', False): continue
                    self.chunks.append(c)
                    self.id_to_idx[c['chunk_id']] = len(self.chunks) - 1

    def _build_art_map(self) -> None:
        # get_entity_id works for both article_number (Constitution)
        # and section_number_raw (all other acts)
        for c in self.chunks:
            a = get_entity_id(c)
            if a: self.art_to_ids.setdefault(a, []).append(c['chunk_id'])

    def _build(self) -> None:
        corpus = []
        for c in tqdm(self.chunks, desc="BM25 tokenise"):
            eid = get_entity_id(c)
            enriched = ' '.join(filter(None, [
                c.get('chunk_text',''), c.get('title',''),
                ' '.join(c.get('keywords',[])),
                ' '.join(c.get('legalsaathi_use_cases',[])),
                c.get('source',''),
                f"Section {eid}" if eid else ''
            ]))
            corpus.append(enriched.lower().split())
        self.bm25 = BM25Okapi(corpus)

indexer = LegalIndexer()
indexer.load()
print("✓ LegalIndexer ready")

  [Skip] consumer_protection_act_chunks.ndjson not found
[BM25] Restoring from Drive cache...
[BM25] Ready — 2,573 chunks
✓ LegalIndexer ready


In [ ]:
# Cell 2.2 — Test BM25
tests = [
    ("right to life personal liberty",       "Art.21 — Constitution"),
    ("police arrested without reason",        "Art.22 — Constitution"),
    ("defective product compensation consumer","Sec.39 — Consumer Protection"),
    ("FIR complaint police investigation",    "Sec.154 — CrPC"),
    ("domestic violence protection order",    "Sec.18 — DV Act"),
]
print("BM25 Test")
print("="*65)
for q, label in tests:
    res = indexer.bm25_search(q, top_k=1)
    if not res: print(f"  ✗  No result: {q}"); continue
    chunk, score = res[0]
    eid = get_entity_id(chunk)
    src = chunk.get('source', '')[:30]
    print(f"Query : {q[:45]:<45}")
    print(f"  → Sec/Art.{eid:<8} [{src}]  BM25={score:.1f}  (expected: {label})")

BM25 Test
Query : right to life personal liberty               
  → Sec/Art.21       [legislative.gov.in]  BM25=33.3  (expected: Art.21 — Constitution)
Query : police arrested without reason               
  → Sec/Art.22       [legislative.gov.in]  BM25=12.2  (expected: Art.22 — Constitution)
Query : defective product compensation consumer      
  → Sec/Art.31A      [legislative.gov.in]  BM25=5.0  (expected: Sec.39 — Consumer Protection)
Query : FIR complaint police investigation           
  → Sec/Art.262      [legislative.gov.in]  BM25=8.6  (expected: Sec.154 — CrPC)
Query : domestic violence protection order           
  → Sec/Art.22       [legislative.gov.in]  BM25=7.6  (expected: Sec.18 — DV Act)


---
## 🧠 Phase 3 — Pinecone Vector Index (Gemini Embeddings)

**`text-embedding-004`** produces **768-dim** vectors (not 1536).  
The Pinecone index `dimension` must be **768**.

**Task types** (Gemini-specific — improves retrieval quality):
- `retrieval_document` → when embedding chunks for storage
- `retrieval_query`    → when embedding a user query at search time

**Free tier rate limit:** 100 requests/min → we add `time.sleep(0.7)` between calls.  
All 603 chunks embed in ~7 minutes on the free tier.


In [ ]:
def gemini_embed(texts: List[str], task_type: str = 'RETRIEVAL_DOCUMENT') -> List[List[float]]:
    response = embed_client.models.embed_content(
        model    = config.EMBED_MODEL,
        contents = texts,
        config   = types.EmbedContentConfig(task_type=task_type)
    )
    time.sleep(13)   # 13s gap → stays under 5 RPM free-tier limit
    return [list(emb.values) for emb in response.embeddings]

In [ ]:
# Cell 3.2 — Delete old 768-dim index and recreate at 3072

from pinecone import Pinecone, ServerlessSpec

pc       = Pinecone(api_key=config.PINECONE_API_KEY)
existing = [i.name for i in pc.list_indexes()]

# Delete the old wrong-dimension index
if config.PINECONE_INDEX in existing:
    print(f"Deleting old index '{config.PINECONE_INDEX}' (wrong dim)...")
    pc.delete_index(config.PINECONE_INDEX)
    print("Deleted.")

# Recreate with correct dimension
print(f"Creating index '{config.PINECONE_INDEX}' (dim={config.EMBED_DIM})...")
pc.create_index(
    name      = config.PINECONE_INDEX,
    dimension = config.EMBED_DIM,   # 3072
    metric    = 'cosine',
    spec      = ServerlessSpec(cloud='aws', region='us-east-1')
)

pinecone_index = pc.Index(config.PINECONE_INDEX)
print(f"✓ Index ready — dim={config.EMBED_DIM}")
print(f"  Vectors: {pinecone_index.describe_index_stats().total_vector_count:,}")

Deleting old index 'legal-saathi-gemini' (wrong dim)...
Deleted.
Creating index 'legal-saathi-gemini' (dim=3072)...
✓ Index ready — dim=3072
  Vectors: 0


In [ ]:
# Cell 3.3 — Embed all chunks + upsert to Pinecone (resumable)
# Safe to re-run anytime — skips chunks already in Pinecone.

all_chunks = indexer.all_chunks()

# Fetch IDs already in Pinecone so we resume exactly where we left off
print("Checking already-indexed chunks...")
already_indexed = set()
try:
    for id_batch in pinecone_index.list(namespace=config.PINECONE_NS):
        for item in id_batch:
            already_indexed.add(item if isinstance(item, str) else item.id)
except Exception as e:
    print(f"  Note: list() failed ({e}) — starting fresh")

remaining = [c for c in all_chunks if c['chunk_id'] not in already_indexed]
print(f"Already indexed : {len(already_indexed):,}")
print(f"Remaining       : {len(remaining):,}")

if not remaining:
    print("✓ All chunks already indexed — nothing to do")
else:
    BATCH = 20
    print(f"Embedding {len(remaining):,} chunks (batch={BATCH})...")
    print(f"API calls needed: ~{(len(remaining) + BATCH - 1) // BATCH} (batch embed = 1 call per batch)")

    for i in tqdm(range(0, len(remaining), BATCH), desc="Upserting"):
        batch = remaining[i : i + BATCH]
        texts = [c['chunk_text'].replace('\n', ' ') for c in batch]

        # Retry loop — handles transient 429s (per-minute limit, not daily)
        while True:
            try:
                embs = gemini_embed(texts, task_type='RETRIEVAL_DOCUMENT')
                break
            except Exception as e:
                err = str(e)
                if '429' in err or 'RESOURCE_EXHAUSTED' in err:
                    if 'PerDay' in err or 'per_day' in err.lower():
                        print(f"\n  ❌  Daily quota exhausted.")
                        print(f"      ~{i * BATCH} chunks indexed so far. Resume tomorrow.")
                        raise   # stops the loop cleanly
                    else:
                        print(f"\n  ⚠️  Per-minute rate limit — waiting 70s...")
                        time.sleep(70)
                else:
                    raise

        vectors = [{
            'id'      : c['chunk_id'],
            'values'  : emb,
            'metadata': {
                'entity_id' : get_entity_id(c),
                'title'     : c.get('title') or '',
                'act'       : c.get('source') or '',
                'namespace' : c.get('namespace') or '',
                'use_cases' : c.get('legalsaathi_use_cases') or [],
            }
        } for c, emb in zip(batch, embs)]

        pinecone_index.upsert(vectors=vectors, namespace=config.PINECONE_NS)

    final = pinecone_index.describe_index_stats().total_vector_count
    print(f"\n✓ Done — {final:,} vectors in Pinecone namespace '{config.PINECONE_NS}'")

Checking already-indexed chunks...
Already indexed : 1,940
Remaining       : 633
Embedding 633 chunks (batch=20)...
API calls needed: ~32 (batch embed = 1 call per batch)


Upserting:   0%|          | 0/32 [00:00<?, ?it/s]


✓ Done — 2,573 vectors in Pinecone namespace 'all-acts'


In [ ]:
stats = pinecone_index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(dimension=3072, total_vector_count=1000, metric='cosine', namespaces=1)


In [ ]:
# Cell 3.4 — Test dense search with Gemini query embedding
def dense_search(query: str, top_k: int = 5) -> List[Dict]:
    vec = gemini_embed([query], task_type='retrieval_query')[0]
    res = pinecone_index.query(
        vector=vec, top_k=top_k,
        namespace=config.PINECONE_NS, include_metadata=True
    )
    chunks = []
    for m in res.matches:
        c = indexer.get_chunk(m.id)
        if c:
            c = c.copy(); c['_score'] = round(m.score, 4); chunks.append(c)
    return chunks

# Paraphrased queries — where dense search shines
tests = [
    "my child cannot attend school",
    "I bought a defective phone, seller refuses refund",
    "boss refuses to pay my salary",
]
print("Dense Search (Gemini embeddings)")
print("="*60)
for q in tests:
    res = dense_search(q, top_k=2)
    print(f"Query: '{q}'")
    for r in res:
        eid = get_entity_id(r)
        src = r.get('source', '')[:25]
        print(f"  Sec/Art.{eid:<6} [{src}]  {r.get('title','')[:40]}  cos={r['_score']:.4f}")
    print()

Dense Search (Gemini embeddings)
Query: 'my child cannot attend school'
  Sec/Art.21A    [legislative.gov.in]  Right to education  cos=0.6326
  Sec/Art.45     [legislative.gov.in]  Provision for early childhood care and e  cos=0.6258

Query: 'I bought a defective phone, seller refuses refund'
  Sec/Art.43     [indiacode.nic.in]  “Illegal”. “Legally bound to do”  cos=0.5999
  Sec/Art.14     [orissa.gov.in / prsindia.]  Finality of orders  cos=0.5998

Query: 'boss refuses to pay my salary'
  Sec/Art.361B   [legislative.gov.in]  Disqualification for appointment on remu  cos=0.6471
  Sec/Art.42     [indiacode.nic.in]  Corporation’s rights when an employer fa  cos=0.6379



---
## 🕸️ Phase 4 — Knowledge Graph (NetworkX)
No API calls. Fully local. Same as before — unchanged by Gemini switch.


In [ ]:
# Cell 4.1 — LegalGraphStore

class LegalGraphStore:
    def __init__(self):
        self.G = nx.DiGraph()
        self.art_to_node: Dict[str, str] = {}

    def load(self, chunks: List[Dict]) -> None:
        self._load_nodes(); self._load_edges(); self._build_art_map()
        xr = self._add_cross_ref(chunks)
        sc = self._add_same_cat(chunks)
        print(f"[Graph] {self.G.number_of_nodes():,} nodes  "
              f"{self.G.number_of_edges():,} edges  "
              f"(+{xr} xref, +{sc} same_cat)")

    def get_related_articles(self, art: str, depth: int = 2) -> List[str]:
        start = self.art_to_node.get(art)
        if not start or start not in self.G: return []
        visited, related = set(), []
        queue = [(start, 0)]
        while queue:
            curr, d = queue.pop(0)
            if curr in visited or d > depth: continue
            visited.add(curr)
            data = self.G.nodes[curr]
            # Handle both Constitution articles and other-act sections
            if data.get('node_type') in ('article', 'section'):
                props = data.get('properties', {})
                a = props.get('article_number') or props.get('section_number')
                if a and a != art: related.append(a)
            for nb in list(self.G.successors(curr)) + list(self.G.predecessors(curr)):
                if nb not in visited: queue.append((nb, d+1))
        return related

    def get_articles_by_use_case(self, uc: str) -> List[str]:
        result = []
        for _, d in self.G.nodes(data=True):
            if d.get('node_type') in ('article', 'section'):
                props = d.get('properties', {})
                if uc in props.get('use_cases', []):
                    a = props.get('article_number') or props.get('section_number')
                    if a: result.append(a)
        return result

    def _load_nodes(self) -> None:
        # Load from ALL act graph-node files
        for path in config.GRAPH_NODES_PATHS:
            if not os.path.exists(path):
                print(f"  [Skip] {path.split('/')[-1]} not found")
                continue
            with open(path,'r',encoding='utf-8') as f:
                for line in f:
                    line=line.strip()
                    if not line: continue
                    n=json.loads(line)
                    self.G.add_node(n['node_id'],node_type=n['node_type'],
                                    label=n['label'],properties=n.get('properties',{}))

    def _load_edges(self) -> None:
        # Load from ALL act graph-edge files
        for path in config.GRAPH_EDGES_PATHS:
            if not os.path.exists(path):
                print(f"  [Skip] {path.split('/')[-1]} not found")
                continue
            with open(path,'r',encoding='utf-8') as f:
                for line in f:
                    line=line.strip()
                    if not line: continue
                    e=json.loads(line)
                    if self.G.has_node(e['source']) and self.G.has_node(e['target']):
                        self.G.add_edge(e['source'],e['target'],relation=e['relation'])

    def _build_art_map(self) -> None:
        # Handle both 'article' (Constitution) and 'section' (all other acts)
        for nid,data in self.G.nodes(data=True):
            if data.get('node_type') in ('article', 'section'):
                props = data.get('properties',{})
                a = props.get('article_number') or props.get('section_number')
                if a:
                    self.art_to_node[a]=nid
                    props.setdefault('use_cases',[])

    def _add_cross_ref(self, chunks: List[Dict]) -> int:
        added=0
        for c in chunks:
            sa = get_entity_id(c)
            if not sa or sa not in self.art_to_node: continue
            src=self.art_to_node[sa]
            self.G.nodes[src]['properties']['use_cases']=c.get('legalsaathi_use_cases',[])
            self.G.nodes[src]['properties']['article_number']=sa
            for ref in c.get('cross_references',[]):
                # Constitution refs: "21", "Article 21"
                # Other-act refs: "CPA_SEC_84" (full node ID already in graph)
                if self.G.has_node(ref):
                    tgt = ref   # direct node ID lookup
                else:
                    ra=ref.replace('Article ','').replace('article ','').strip()
                    tgt=self.art_to_node.get(ra)
                if tgt and not self.G.has_edge(src,tgt):
                    self.G.add_edge(src,tgt,relation='CROSS_REFERENCES'); added+=1
        return added

    def _add_same_cat(self, chunks: List[Dict]) -> int:
        uc_map: Dict[str,List[str]]={}
        for c in chunks:
            a=get_entity_id(c)
            if not a: continue
            for uc in c.get('legalsaathi_use_cases',[]):
                uc_map.setdefault(uc,[]).append(a)
        added=0
        for uc,arts in uc_map.items():
            unique=list(set(arts))[:8]
            for i in range(len(unique)):
                for j in range(i+1,len(unique)):
                    s=self.art_to_node.get(unique[i])
                    t=self.art_to_node.get(unique[j])
                    if s and t and not self.G.has_edge(s,t):
                        self.G.add_edge(s,t,relation=f'SAME_{uc.upper()}'); added+=1
        return added

graph_store = LegalGraphStore()
graph_store.load(indexer.all_chunks())
print("✓ Graph ready")

  [Skip] consumer_protection_act_graph_nodes.ndjson not found
  [Skip] consumer_protection_act_graph_edges.ndjson not found
[Graph] 2,169 nodes  7,110 edges  (+436 xref, +984 same_cat)
✓ Graph ready


In [ ]:
# Cell 4.2 — Test graph traversal
print("Graph Traversal (depth=2)")
print("="*60)
for art, name in [("21","Right to Life"),("32","Writ Remedy"),("39A","Free Legal Aid")]:
    related = graph_store.get_related_articles(art, depth=2)
    print(f"Art.{art} ({name}) → {len(related)} related: {related[:6]}")

Graph Traversal (depth=2)
Art.21 (Right to Life) → 145 related: ['19', '20', '21A', '22', '32', '226']
Art.32 (Writ Remedy) → 119 related: ['31', '13', '21', '226', '28', '36']
Art.39A (Free Legal Aid) → 120 related: ['39', '32', '226', '44', '41', '42']


---
## ⚡ Phase 5 — Hybrid Retriever (BM25 + Dense + RRF)
Only change from OpenAI version: the embedding call uses `gemini_embed()` with `task_type='retrieval_query'`.


In [ ]:
# Cell 5.1 — HybridRetriever

class HybridRetriever:
    def __init__(self, indexer: LegalIndexer):
        self.indexer   = indexer
        self._pc_index = None
        self.k         = 60   # RRF constant

    def connect(self, pc_index) -> None:
        self._pc_index = pc_index

    def retrieve(self, query: str, top_k: int = None,
                 filter_uc: Optional[str] = None) -> List[Dict]:
        top_k       = top_k or config.FINAL_TOP_K
        bm25_ranked = [c for c, _ in self.indexer.bm25_search(query, config.BM25_TOP_K)]
        dense_ranked = self._dense(query, config.DENSE_TOP_K, filter_uc)
        return self._rrf(bm25_ranked, dense_ranked)[:top_k]

    def embed_query(self, text: str) -> List[float]:
        # task_type='retrieval_query' for queries (not documents)
        return gemini_embed([text], task_type='retrieval_query')[0]

    def _dense(self, q: str, k: int, filter_uc: Optional[str]) -> List[Dict]:
        if not self._pc_index: return []
        vec = self.embed_query(q)
        pf  = {'use_cases': {'$in': [filter_uc]}} if filter_uc else None
        res = self._pc_index.query(vector=vec, top_k=k, namespace=config.PINECONE_NS,
                                   include_metadata=True, filter=pf)
        chunks = []
        for m in res.matches:
            c = self.indexer.get_chunk(m.id)
            if c:
                c = c.copy(); c['_dense_score'] = round(m.score, 4); chunks.append(c)
        return chunks

    def _rrf(self, a: List[Dict], b: List[Dict]) -> List[Dict]:
        scores, cmap = {}, {}
        for rank, c in enumerate(a, 1):
            cid=c['chunk_id']
            scores[cid]=scores.get(cid,0.0)+1.0/(self.k+rank); cmap[cid]=c
        for rank, c in enumerate(b, 1):
            cid=c['chunk_id']
            scores[cid]=scores.get(cid,0.0)+1.0/(self.k+rank); cmap[cid]=c
        result=[]
        for cid in sorted(scores, key=lambda x:scores[x], reverse=True):
            c=cmap[cid].copy(); c['_rrf_score']=round(scores[cid],6); c['_source']='hybrid'
            result.append(c)
        return result

hybrid = HybridRetriever(indexer)
hybrid.connect(pinecone_index)
print("✓ HybridRetriever ready")


✓ HybridRetriever ready


In [ ]:
# Cell 5.2 — Test hybrid retrieval
tests = [
    ("Article 21A right to education",    "keyword — BM25 wins"),
    ("my child cannot attend school",     "paraphrase — Dense wins"),
    ("police arrested me without reason", "mix of both"),
]
print("Hybrid Results")
print("="*65)
for q, note in tests:
    res = hybrid.retrieve(q, top_k=3)
    print(f"Query : {q}")
    print(f"Note  : {note}")
    for r in res:
        print(f"  Art.{r.get('article_number','?'):<5} {r.get('title','')[:45]:<45} RRF={r.get('_rrf_score',0):.6f}")
    print()


Hybrid Results
Query : Article 21A right to education
Note  : keyword — BM25 wins
  Art.21A   Right to education                            RRF=0.032787
  Art.41    Right to work, to education and to public ass RRF=0.031514
  Art.45    Provision for early childhood care and educat RRF=0.031514

Query : my child cannot attend school
Note  : paraphrase — Dense wins
  Art.28    Freedom as to attendance at religious instruc RRF=0.031250
  Art.24    Prohibition of employment of children in fact RRF=0.031099
  Art.21A   Right to education                            RRF=0.016393

Query : police arrested me without reason
Note  : mix of both
  Art.22    Protection against arrest and detention in ce RRF=0.032018
  Art.?     When police may arrest without warrant        RRF=0.016393
  Art.312   All-India services                            RRF=0.016129



---
## 🔗 Phase 6 — Graph RAG Retriever
No API calls — fully local graph traversal. Unchanged from OpenAI version.


In [ ]:
# Cell 6.1 — GraphRetriever + test

class GraphRetriever:
    def __init__(self, graph: LegalGraphStore, indexer: LegalIndexer):
        self.graph=graph; self.indexer=indexer

    def expand(self, initial: List[Dict], depth: int=2, max_add: int=5) -> List[Dict]:
        seen={c['chunk_id'] for c in initial}; extra=[]
        for chunk in initial:
            art=get_entity_id(chunk)   # works for both article_number and section_number_raw
            if not art: continue
            for ra in self.graph.get_related_articles(art, depth=depth):
                if len(extra)>=max_add: break
                for rc in self.indexer.get_chunks_for_article(ra):
                    if rc['chunk_id'] not in seen:
                        rc=rc.copy(); rc['_source']='graph'; rc['_graph_origin']=art
                        extra.append(rc); seen.add(rc['chunk_id'])
        print(f"[Graph] Added {len(extra)} graph-discovered chunks")
        return initial+extra

graph_ret = GraphRetriever(graph_store, indexer)

# Test
query   = "Can I get legal help if I cannot afford a lawyer?"
initial  = hybrid.retrieve(query, top_k=3)
expanded = graph_ret.expand(initial, depth=2, max_add=4)
print(f"Query: '{query}'")
print(f"Hybrid: {len(initial)} → After graph expansion: {len(expanded)}")
for c in expanded:
    icon = "🕸️" if c.get('_source')=='graph' else "⚡"
    orig = f" via {c['_graph_origin']}" if c.get('_graph_origin') else ""
    eid  = get_entity_id(c)
    src  = c.get('source','')[:20]
    print(f"  {icon} Sec/Art.{eid}: {c.get('title','')[:40]}  [{src}]{orig}")

[Graph] Added 9 graph-discovered chunks
Query: 'Can I get legal help if I cannot afford a lawyer?'
Hybrid: 3 → After graph expansion: 12
  ⚡ Sec/Art.39A: Equal justice and free legal aid  [legislative.gov.in]
  ⚡ Sec/Art.69: Oath or affirmation by the Vice-Presiden  [legislative.gov.in]
  ⚡ Sec/Art.304: Legal aid to accused at State expense in  [indiacode.nic.in]
  🕸️ Sec/Art.39: Certain principles of policy to be follo  [legislative.gov.in] via 39A
  🕸️ Sec/Art.39: Public to give information of certain of  [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Public to give information of certain of  [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Illustration

 A sets fire, by night, to  [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Guidelines for child to take assistance   [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Medical benefit  [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Medical benefit  [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Medical benefit  [indiacode.nic.in] via 39A
  🕸️ Sec/Art.39: Ap

---
## 🔁 Phase 7 — Multi-hop RAG
Gap detection now uses `gemini-1.5-flash` instead of `gpt-4o-mini`.  
The prompt is identical — only the API call changes.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 7 — Multi-hop RAG
# ══════════════════════════════════════════════════════════════════════════════

# Cell 7.1 — MultiHopRetriever

class MultiHopRetriever:
    def __init__(self, hybrid: HybridRetriever):
        self.hybrid = hybrid

    def retrieve(self, query: str, max_hops: int = None) -> List[Dict]:
        max_hops    = max_hops or config.MULTIHOP_ITERS
        all_chunks  : List[Dict] = []
        seen        : Set[str]   = set()
        curr_q      = query

        for hop in range(1, max_hops + 1):
            print(f"  Hop {hop}: '{curr_q[:60]}'")
            new = self.hybrid.retrieve(curr_q, top_k=5)
            added = 0
            for c in new:
                if c['chunk_id'] not in seen:
                    c = c.copy(); c['_hop'] = hop
                    all_chunks.append(c); seen.add(c['chunk_id']); added += 1
            print(f"  → {added} new  (total={len(all_chunks)})")

            if hop == max_hops:
                break

            nq = self._detect_gap(query, curr_q, all_chunks)
            if nq is None:
                print(f"  → No gap — stopping")
                break
            curr_q = nq

        return all_chunks

    def _detect_gap(self, original: str, last_q: str, chunks: List[Dict]) -> Optional[str]:
        summary = "\n".join(
            f"- Sec/Art.{get_entity_id(c)} [{c.get('source','')}]: {c.get('title','Untitled')}"
            for c in chunks[:12]
        )
        prompt = (
            f'Original legal question: "{original}"\n\n'
            f'Laws retrieved so far:\n{summary}\n\n'
            f'Is there an Indian law provision still needed to fully answer the question?\n'
            f'YES → reply with ONLY a short search query (under 10 words).\n'
            f'NO  → reply with ONLY the word: COMPLETE'
        )
        resp   = groq_client.chat.completions.create(
            model    = config.LLM_MODEL,
            messages = [{"role": "user", "content": prompt}]
        )
        result = resp.choices[0].message.content.strip()
        if result.upper() == 'COMPLETE' or not result or result.lower() == last_q.lower():
            return None
        print(f"  → Gap found: '{result}'")
        return result

multi_hop = MultiHopRetriever(hybrid)   # ← no indentation
print("✓ MultiHopRetriever ready")

✓ MultiHopRetriever ready


In [ ]:
multi_hop = MultiHopRetriever(hybrid)
print("✓ MultiHopRetriever ready")

✓ MultiHopRetriever ready


In [ ]:
# Cell 7.2 — Test multi-hop
q = "My son was arrested 4 days ago, no FIR filed, still in custody"
print(f"Query: '{q}'\n")
mhc = multi_hop.retrieve(q, max_hops=3)

print("\nResults by hop:")
for h in range(1, 4):
    hc = [c for c in mhc if c.get('_hop') == h]
    if hc:
        arts = [f"Art.{c.get('article_number')}" for c in hc]   # ✅ no nested quotes issue
        print(f"  Hop {h}: {arts}")

Query: 'My son was arrested 4 days ago, no FIR filed, still in custody'

  Hop 1: 'My son was arrested 4 days ago, no FIR filed, still in custo'
  → 5 new  (total=5)
  → Gap found: 'Sec 167 CrPC'
  Hop 2: 'Sec 167 CrPC'
  → 5 new  (total=10)
  → Gap found: 'Section 167 details'
  Hop 3: 'Section 167 details'
  → 3 new  (total=13)

Results by hop:
  Hop 1: ['Art.22', 'Art.None', 'Art.284', 'Art.None', 'Art.283']
  Hop 2: ['Art.167', 'Art.None', 'Art.None', 'Art.None', 'Art.None']
  Hop 3: ['Art.372', 'Art.None', 'Art.347']


---
## 🤖 Phase 8 — LLM Generation + Hallucination Validator

**Gemini JSON mode:** Pass `response_mime_type="application/json"` in `GenerationConfig`.  
This is Gemini's equivalent of OpenAI's `response_format={"type":"json_object"}`.  
Guarantees valid JSON output — no regex parsing needed.

**Validator is identical** — it's pure Python, no API calls.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 8 — LLM Generation + Hallucination Validator
# ══════════════════════════════════════════════════════════════════════════════

# Cell 8.1 — LegalGenerator

SYSTEM_PROMPT = """You are LegalSaathi, a helpful Indian legal assistant.
Given a user's legal problem and retrieved sections/articles from Indian statutes:
1. Explain each relevant law at a Class 8 reading level — no legal jargon.
2. Give one real-life example per law.
3. List 2-3 concrete action steps.
4. Respond in the SAME LANGUAGE as the user's query.
5. ONLY cite sections/articles in the provided context. Never invent section numbers.

Respond ONLY in this JSON format:
{
  "law_cards": [{
    "article_number": "84",
    "act": "Consumer Protection Act, 2019",
    "title": "Liability of product manufacturer",
    "explanation": "...",
    "example": "...",
    "action_steps": ["Step 1", "Step 2"]
  }]
}"""


class LegalGenerator:

    def generate(self, query: str, chunks: List[Dict], lang: str = 'en') -> Dict:
        context = "\n\n".join(
            f"--- {c.get('source', 'Indian Law')} | Sec/Art.{get_entity_id(c)} | {c.get('title','')} ---\n"
            f"{c.get('chunk_text', c.get('body_text',''))}"
            for c in chunks
        )
        lang_str = 'Hindi' if lang == 'hi' else 'English'
        prompt   = (
            f'Legal problem: "{query}"\n\nContext:\n{context}\n\n'
            f'Generate law cards in {lang_str}. Only use sections/articles in the context above.'
        )

        print(f"[Gen] Calling {config.LLM_MODEL}...")

        resp = groq_client.chat.completions.create(
        model    = config.LLM_MODEL,
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt}
        ],
        temperature = 0.1
        )
        return self._validate(resp.choices[0].message.content, chunks)

    def _validate(self, raw: str, chunks: List[Dict]) -> Dict:
        try:
            parsed = json.loads(raw)
        except:
            parsed = {'law_cards': []}

        # Build valid set using get_entity_id — works for both article & section chunks
        valid  = {str(get_entity_id(c)).strip()
                  for c in chunks if get_entity_id(c)}
        cards, removed = [], 0

        for card in parsed.get('law_cards', []):
            # LLM may use 'article_number' or 'section_number' in its response
            art = str(card.get('article_number', '') or
                      card.get('section_number', '')).strip()
            if art in valid:
                if not isinstance(card.get('action_steps'), list):
                    card['action_steps'] = []
                cards.append(card)
            else:
                print(f"  ⚠️  Hallucination removed: {art}")
                removed += 1

        if removed:
            print(f"[Validator] Removed {removed} hallucinated citation(s)")
        print(f"[Gen] ✓ {len(cards)} validated law cards")
        return {'law_cards': cards}


generator = LegalGenerator()
print("✓ LegalGenerator ready")

✓ LegalGenerator ready


---
## 🏛️ Phase 9 — Full Pipeline + Tests

Same routing logic as before. Only the underlying API calls use Gemini.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 9 — Full Pipeline + Tests
# ══════════════════════════════════════════════════════════════════════════════

# Cell 9.1 — Language detect + Query classify + Pipeline

try:
    from langdetect import detect
    LANGDETECT_OK = True
except:
    LANGDETECT_OK = False


def detect_language(text: str) -> str:
    if not LANGDETECT_OK:
        return 'en'
    try:
        lang = detect(text)
        return lang if lang in ('en', 'hi') else 'en'
    except:
        return 'en'


def classify_query(query: str) -> str:
    prompt = (
        'Classify this Indian legal query. Reply with ONE word only.\n'
        'SIMPLE = one article/section or definition\n'
        'RELATIONAL = how laws relate to each other\n'
        'COMPLEX = real-life situation needing multiple laws or procedures\n\n'
        f'Query: "{query}"\n'
        'Answer (SIMPLE / RELATIONAL / COMPLEX):'
    )
    resp = groq_client.chat.completions.create(
    model    = config.LLM_MODEL,
    messages = [{"role": "user", "content": prompt}]
    )
    r = resp.choices[0].message.content.strip().upper()
    if 'RELATIONAL' in r: return 'relational'
    if 'COMPLEX'    in r: return 'complex'
    return 'simple'


def compress(chunks: List[Dict]) -> List[Dict]:
    total, out = 0, []
    for c in chunks:
        n = len(c.get('chunk_text', ''))
        if total + n > config.MAX_CONTEXT_CHARS:
            break
        out.append(c); total += n
    print(f"[Compress] {len(out)}/{len(chunks)} chunks  ({total:,} chars)")
    return out


def lawyer_category(chunks: List[Dict]) -> str:
    counts = {}
    for c in chunks:
        for uc in c.get('legalsaathi_use_cases', []):
            counts[uc] = counts.get(uc, 0) + 1
    if not counts:
        return 'general'
    top = max(counts, key=counts.get)
    # Expanded map covers all 10 acts' use cases
    return {
        # Constitutional
        'fundamental'           : 'constitutional',
        'writ'                  : 'constitutional',
        'constitutional_remedy' : 'constitutional',
        'education'             : 'constitutional',
        'dpsp'                  : 'constitutional',
        # Criminal
        'criminal'              : 'criminal',
        'arrest'                : 'criminal',
        'fir'                   : 'criminal',
        'bail'                  : 'criminal',
        'investigation'         : 'criminal',
        'child_protection'      : 'criminal',
        'sexual_offence'        : 'criminal',
        # Family
        'family_law'            : 'family',
        'domestic_violence'     : 'family',
        'marriage'              : 'family',
        'divorce'               : 'family',
        'maintenance'           : 'family',
        'personal_law'          : 'family',
        # Labour
        'labour'                : 'labour',
        'workplace_rights'      : 'labour',
        'social_security'       : 'labour',
        'gratuity'              : 'labour',
        'pf'                    : 'labour',
        'employment'            : 'labour',
        # Consumer
        'consumer_complaint'    : 'consumer',
        'consumer_rights'       : 'consumer',
        'product_liability'     : 'consumer',
        'product_defect'        : 'consumer',
        'service_deficiency'    : 'consumer',
        'refund_compensation'   : 'consumer',
        'e_commerce_dispute'    : 'consumer',
        'unfair_trade_practice' : 'consumer',
        'misleading_advertisement': 'consumer',
        'district_forum'        : 'consumer',
        'state_commission'      : 'consumer',
        'national_commission'   : 'consumer',
        'mediation'             : 'consumer',
        # Civil / Property
        'property'              : 'civil',
        'housing'               : 'civil',
        'rent'                  : 'civil',
        'landlord_tenant'       : 'civil',
    }.get(top, 'general')


class LegalSaathiPipeline:
    def __init__(self, indexer, graph_store, hybrid, graph_ret, multi_hop, generator):
        self.indexer  = indexer
        self.gs       = graph_store
        self.hybrid   = hybrid
        self.gr       = graph_ret
        self.mh       = multi_hop
        self.gen      = generator

    def query(self, user_query: str) -> Dict:
        print(f'\n{"="*60}')
        print(f'Query: {user_query[:70]}')

        lang  = detect_language(user_query)
        qtype = classify_query(user_query)
        print(f'Language: {lang}  |  Type: {qtype}')

        if qtype == 'simple':
            chunks = self.hybrid.retrieve(user_query, config.FINAL_TOP_K)

        elif qtype == 'relational':
            ini    = self.hybrid.retrieve(user_query, 6)
            chunks = self.gr.expand(ini, 2, 4)[:config.FINAL_TOP_K]

        else:  # complex
            mhc    = self.mh.retrieve(user_query, config.MULTIHOP_ITERS)
            chunks = self.gr.expand(mhc, 1, 3)[:config.FINAL_TOP_K]

        chunks = compress(chunks)
        result = self.gen.generate(user_query, chunks, lang)

        return {
            'query'           : user_query,
            'language'        : lang,
            'query_type'      : qtype,
            'law_cards'       : result['law_cards'],
            'retrieved_chunks': len(chunks),
            'lawyer_category' : lawyer_category(chunks),
            'disclaimer'      : 'LegalSaathi provides legal awareness only. Consult a registered lawyer.',
            'sources'         : [
                {'article': get_entity_id(c),
                 'title'  : c.get('title', ''),
                 'act'    : c.get('source', ''),
                 'source' : c.get('_source', 'hybrid')}
                for c in chunks
            ]
        }


pipeline = LegalSaathiPipeline(
    indexer, graph_store, hybrid, graph_ret, multi_hop, generator
)
print("✓ Pipeline ready")

✓ Pipeline ready


In [ ]:
GROQ_KEY = userdata.get('GROQ_API_KEY')
print(f"Key loaded: {'✓' if GROQ_KEY else '✗ MISSING'}")
print(f"Key preview: {GROQ_KEY[:8]}..." if GROQ_KEY else "No key found")

Key loaded: ✓
Key preview: gsk_tkpU...


In [ ]:
# Cell 9.2 — Print helper
def show(r):
    print(f"Language : {r['language']}  |  Type: {r['query_type']}  |  Lawyer: {r['lawyer_category']}")
    print(f"Sources  : {[(s['article'], s['act'][:20]) for s in r['sources'][:4]]}")
    for card in r['law_cards']:
        num = card.get('article_number') or card.get('section_number', '?')
        print(f"\n  Sec/Art.{num} [{card.get('act','')}]: {card['title']}")
        print(f"  {card['explanation'][:200]}...")
        print(f"  Steps: {card['action_steps'][:2]}")
    print(f"\n  {r['disclaimer']}")

In [ ]:
# Cell 9.3 — Test: SIMPLE
r1 = pipeline.query("What is the right to life under Article 21?")
show(r1)



Query: What is the right to life under Article 21?
Language: en  |  Type: simple
[Compress] 8/8 chunks  (5,136 chars)
[Gen] Calling llama-3.3-70b-versatile...
[Gen] ✓ 1 validated law cards
Language : en  |  Type: simple  |  Lawyer: constitutional
Sources  : [('21', 'legislative.gov.in'), ('21A', 'legislative.gov.in'), ('48A', 'legislative.gov.in'), ('22', 'legislative.gov.in')]

  Sec/Art.21 [The Constitution of India]: Right to Life and Personal Liberty
  This law says that no one can be killed or have their freedom taken away unless it is done according to the rules and procedures set by the law. It means that the government cannot take away someone's...
  Steps: ['If you or someone you know is being detained without reason, contact a lawyer immediately.', 'File a complaint with the nearest police station or court if you believe your right to life and personal liberty is being violated.']

  LegalSaathi provides legal awareness only. Consult a registered lawyer.


In [ ]:
# Cell 9.4 — Test: COMPLEX
r2 = pipeline.query(
    "My landlord threw me out at midnight with no notice after 8 years. "
    "What are my constitutional rights and what can I do?"
)
show(r2)


Query: My landlord threw me out at midnight with no notice after 8 years. Wha
Language: en  |  Type: complex
  Hop 1: 'My landlord threw me out at midnight with no notice after 8 '
  → 5 new  (total=5)
  → Gap found: 'Eviction laws India'
  Hop 2: 'Eviction laws India'
  → 4 new  (total=9)
  → Gap found: 'Eviction laws in India'
  Hop 3: 'Eviction laws in India'
  → 2 new  (total=11)
[Graph] Added 5 graph-discovered chunks
[Compress] 8/8 chunks  (5,399 chars)
[Gen] Calling llama-3.3-70b-versatile...
[Gen] ✓ 3 validated law cards
Language : en  |  Type: complex  |  Lawyer: general
Sources  : [('60', 'legislative.gov.in'), ('6', 'orissa.gov.in / prsi'), ('159', 'legislative.gov.in'), ('300A', 'legislative.gov.in')]

  Sec/Art.300A [The Constitution of India]: Protection from Eviction without Authority of Law
  This article means that no one can be thrown out of their property without following the proper legal process. In your case, being evicted at midnight without notice is against th

In [ ]:
# Cell 9.5 — Test: HINDI
r3 = pipeline.query("मुझे पुलिस ने बिना कारण गिरफ्तार किया। मेरे क्या अधिकार हैं?")
show(r3)



Query: मुझे पुलिस ने बिना कारण गिरफ्तार किया। मेरे क्या अधिकार हैं?
Language: hi  |  Type: complex
  Hop 1: 'मुझे पुलिस ने बिना कारण गिरफ्तार किया। मेरे क्या अधिकार हैं?'
  → 5 new  (total=5)
  → Gap found: 'Sec 55 IPC'
  Hop 2: 'Sec 55 IPC'
  → 5 new  (total=10)
  → Gap found: 'Sec 357 CrPC India'
  Hop 3: 'Sec 357 CrPC India'
  → 5 new  (total=15)
[Graph] Added 14 graph-discovered chunks
[Compress] 8/8 chunks  (7,169 chars)
[Gen] Calling llama-3.3-70b-versatile...
[Gen] ✓ 4 validated law cards
Language : hi  |  Type: complex  |  Lawyer: criminal
Sources  : [('50', 'indiacode.nic.in'), ('41', 'indiacode.nic.in'), ('22', 'legislative.gov.in'), ('41', 'indiacode.nic.in')]

  Sec/Art.22 [भारतीय संविधान]: गिरफ्तारी और निरोध से संरक्षण
  किसी भी व्यक्ति को गिरफ्तार करने के बाद, पुलिस को उसे गिरफ्तारी के कारणों के बारे में बताना होगा और उसे एक वकील से परामर्श करने का अधिकार होगा। गिरफ्तार व्यक्ति को 24 घंटे के भीतर निकटतम मजिस्ट्रेट के...
  Steps: ['मजिस्ट्रेट के सामने अपनी बात रखें', 'एक 